In [1]:
pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install implicit

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd 
import numpy as np 
import warnings
import ast
from rapidfuzz import process
from scipy.sparse import csr_matrix
from collections import defaultdict

import implicit

warnings.filterwarnings('ignore')

pd.set_option('display.max_rows',100)

In [4]:
df_user = pd.read_csv('user_data.csv')
df_user.head(10)

,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Lissie,All Be Okay,HARD ROCK 2010
5,9cc0cfd4d7d7885102480dd99e7a90d6,Paul McCartney,Band On The Run,HARD ROCK 2010
6,9cc0cfd4d7d7885102480dd99e7a90d6,Paul McCartney,"Blackbird - Live at CitiField, NYC - Digital A...",HARD ROCK 2010
7,9cc0cfd4d7d7885102480dd99e7a90d6,Lissie,Bright Side,HARD ROCK 2010
8,9cc0cfd4d7d7885102480dd99e7a90d6,Paul McCartney,Dance Tonight,HARD ROCK 2010
9,9cc0cfd4d7d7885102480dd99e7a90d6,Crowded House,Don't Dream It's Over,HARD ROCK 2010


In [5]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8549544 entries, 0 to 8549543
Data columns (total 4 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   user_id       object
 1   artistname    object
 2   trackname     object
 3   playlistname  object
dtypes: object(4)
memory usage: 260.9+ MB


In [6]:
print(df_user.shape)
print(df_user.head(10))
print(df_user.duplicated(subset=['user_id', 'trackname']).sum())

(8549544, 4)
                            user_id                        artistname  \
0  9cc0cfd4d7d7885102480dd99e7a90d6                    Elvis Costello   
1  9cc0cfd4d7d7885102480dd99e7a90d6  Elvis Costello & The Attractions   
2  9cc0cfd4d7d7885102480dd99e7a90d6  Elvis Costello & The Attractions   
3  9cc0cfd4d7d7885102480dd99e7a90d6                    Elvis Costello   
4  9cc0cfd4d7d7885102480dd99e7a90d6                            Lissie   
5  9cc0cfd4d7d7885102480dd99e7a90d6                    Paul McCartney   
6  9cc0cfd4d7d7885102480dd99e7a90d6                    Paul McCartney   
7  9cc0cfd4d7d7885102480dd99e7a90d6                            Lissie   
8  9cc0cfd4d7d7885102480dd99e7a90d6                    Paul McCartney   
9  9cc0cfd4d7d7885102480dd99e7a90d6                     Crowded House   

                                           trackname    playlistname  
0               (The Angels Wanna Wear My) Red Shoes  HARD ROCK 2010  
1  (What's So Funny 'Bout) Peace, Love An

In [7]:
df_master = pd.read_csv('master_track_features.csv')
df_master.head(10)

,id,key,artist_main,name,genres,popularity,valence,year,acousticness,danceability,...,instrumentalness_genre,liveness_genre,loudness_genre,speechiness_genre,tempo_genre,valence_genre,popularity_genre,key_genre,mode_genre,count
0,4BJqT0PrAfrxzMOxytFOIz,10,Sergei Rachmaninoff,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...","['classical', 'post-romantic era', 'russian ro...",4,0.0594,1921,0.982,0.279,...,0.793356,0.236443,-20.485082,0.041968,95.200198,0.264284,4.332090,2.0,1.0,268.0
1,4BJqT0PrAfrxzMOxytFOIz,10,James Levine,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...","['classical performance', 'opera', 'orchestral...",4,0.0594,1921,0.982,0.279,...,0.446163,0.232672,-19.491500,0.048306,102.657000,0.165772,25.833333,10.0,1.0,18.0
2,4BJqT0PrAfrxzMOxytFOIz,10,Berliner Philharmoniker,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...","['classical', 'classical performance', 'german...",4,0.0594,1921,0.982,0.279,...,0.746398,0.203505,-20.868388,0.044848,100.722912,0.164508,26.923529,2.0,1.0,170.0
3,7xPhfUan2yNtyFG0cUWkt8,7,Dennis Day,Clancy Lowered the Boom,[],5,0.9630,1921,0.732,0.819,...,0.000002,0.259680,-11.580400,0.132540,95.256400,0.640600,3.000000,7.0,1.0,10.0
4,1o6I8BglA6ylDMrIELygv1,3,KHP Kridhamardawa Karaton Ngayogyakarta Hadini...,Gati Bali,[],5,0.0394,1921,0.961,0.328,...,0.498736,0.148443,-17.178000,0.053462,108.157619,0.101700,2.428571,9.0,1.0,42.0
5,3ftBPsC5vPBKxYSee08FDH,5,Frank Parker,Danny Boy,[],3,0.1650,1921,0.967,0.275,...,0.000028,0.381000,-9.316000,0.035400,100.109000,0.165000,3.000000,5.0,1.0,2.0
6,4d6HGyGT8e121BsdKmw9v6,3,Phil Regan,When Irish Eyes Are Smiling,[],2,0.2530,1921,0.957,0.418,...,0.000216,0.195000,-11.083000,0.036000,93.718667,0.220000,1.333333,3.0,1.0,6.0
7,4pyw9DVHGStUre4J6hPngr,2,KHP Kridhamardawa Karaton Ngayogyakarta Hadini...,Gati Mardika,[],6,0.1960,1921,0.579,0.697,...,0.498736,0.148443,-17.178000,0.053462,108.157619,0.101700,2.428571,9.0,1.0,42.0
8,5uNZnElqOS3W4fRmRYPk4T,0,John McCormack,The Wearing of the Green,"['irish ballad', 'vintage classical singing']",4,0.4060,1921,0.996,0.518,...,0.000011,0.107400,-13.036000,0.060380,115.056600,0.287200,1.600000,7.0,1.0,5.0
9,02GDntOXexBFUvSgaXLPkd,1,Sergei Rachmaninoff,"Morceaux de fantaisie, Op. 3: No. 2, Prélude i...","['classical', 'post-romantic era', 'russian ro...",2,0.0731,1921,0.993,0.389,...,0.793356,0.236443,-20.485082,0.041968,95.200198,0.264284,4.332090,2.0,1.0,268.0


In [8]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8549544 entries, 0 to 8549543
Data columns (total 4 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   user_id       object
 1   artistname    object
 2   trackname     object
 3   playlistname  object
dtypes: object(4)
memory usage: 260.9+ MB


In [11]:
# # Ambil sample random 500 baris
# sample_df = df_user.sample(n=500, random_state=42)

# # Reset index biar rapi
# sample_df = sample_df.reset_index(drop=True)

# # Simpan ke file baru (opsional)
# sample_df.to_csv("sample_user.csv", index=False)

# # Preview
# print(sample_df.head())
# print(f"Jumlah data sample: {len(sample_df)}")

In [12]:
# # Ambil sample random 500 baris
# sample_df = df_master.sample(n=500, random_state=42)

# # Reset index biar rapi
# sample_df = sample_df.reset_index(drop=True)

# # Simpan ke file baru (opsional)
# sample_df.to_csv("sample_master.csv", index=False)

# # Preview
# print(sample_df.head())
# print(f"Jumlah data sample: {len(sample_df)}")

### **Cleaning df_master**

In [9]:
# Standardisasi nama untuk matching
df_master['name_clean'] = df_master['name'].str.lower().str.strip()
df_master['artist_clean'] = df_master['artist_main'].str.lower().str.strip()

# Parse genres dari string list ke list Python
df_master['genres_list'] = df_master['genres'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

# Ambil genre utama (genre pertama)
df_master['primary_genre'] = df_master['genres_list'].apply(
    lambda x: x[0] if len(x) > 0 else 'unknown'
)

# Kolom audio features yang akan dipakai
audio_features = [
    'valence', 'acousticness', 'danceability', 'energy',
    'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo'
]

In [10]:
df_master.head()

,id,key,artist_main,name,genres,popularity,valence,year,acousticness,danceability,...,tempo_genre,valence_genre,popularity_genre,key_genre,mode_genre,count,name_clean,artist_clean,genres_list,primary_genre
0,4BJqT0PrAfrxzMOxytFOIz,10,Sergei Rachmaninoff,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...","['classical', 'post-romantic era', 'russian ro...",4,0.0594,1921,0.982,0.279,...,95.200198,0.264284,4.332090,2.0,1.0,268.0,"piano concerto no. 3 in d minor, op. 30: iii. ...",sergei rachmaninoff,"[classical, post-romantic era, russian romanti...",classical
1,4BJqT0PrAfrxzMOxytFOIz,10,James Levine,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...","['classical performance', 'opera', 'orchestral...",4,0.0594,1921,0.982,0.279,...,102.657000,0.165772,25.833333,10.0,1.0,18.0,"piano concerto no. 3 in d minor, op. 30: iii. ...",james levine,"[classical performance, opera, orchestral perf...",classical performance
2,4BJqT0PrAfrxzMOxytFOIz,10,Berliner Philharmoniker,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...","['classical', 'classical performance', 'german...",4,0.0594,1921,0.982,0.279,...,100.722912,0.164508,26.923529,2.0,1.0,170.0,"piano concerto no. 3 in d minor, op. 30: iii. ...",berliner philharmoniker,"[classical, classical performance, german orch...",classical
3,7xPhfUan2yNtyFG0cUWkt8,7,Dennis Day,Clancy Lowered the Boom,[],5,0.9630,1921,0.732,0.819,...,95.256400,0.640600,3.000000,7.0,1.0,10.0,clancy lowered the boom,dennis day,[],unknown
4,1o6I8BglA6ylDMrIELygv1,3,KHP Kridhamardawa Karaton Ngayogyakarta Hadini...,Gati Bali,[],5,0.0394,1921,0.961,0.328,...,108.157619,0.101700,2.428571,9.0,1.0,42.0,gati bali,khp kridhamardawa karaton ngayogyakarta hadini...,[],unknown


### **Cleaning df_user**

In [11]:
# Standardisasi
df_user['artist_clean'] = df_user['artistname'].str.lower().str.strip()
df_user['track_clean'] = df_user['trackname'].str.lower().str.strip()

# Hitung berapa kali user mendengar artist tertentu
df_user['play_count'] = df_user.groupby(
    ['user_id', 'artistname', 'trackname']
)['artistname'].transform('count')

# Buat implicit rating: lebih sering = lebih tinggi skor
user_artist_matrix = df_user.groupby(['user_id', 'artistname']).size().reset_index(name='implicit_rating')

In [12]:
df_user.head()

,user_id,artistname,trackname,playlistname,artist_clean,track_clean,play_count
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elvis costello,(the angels wanna wear my) red shoes,1.0
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elvis costello & the attractions,"(what's so funny 'bout) peace, love and unders...",1.0
2,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elvis costello & the attractions,accidents will happen,1.0
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elvis costello,alison,1.0
4,9cc0cfd4d7d7885102480dd99e7a90d6,Lissie,All Be Okay,HARD ROCK 2010,lissie,all be okay,1.0


In [13]:
# Cek apakah ada play_count yang lebih dari 1
print(df_user['play_count'].value_counts().head(10))

# Cek user yang punya play_count tinggi
print(df_user[df_user['play_count'] > 1].head(10))

play_count
1.0     6706020
2.0     1347920
3.0      331710
4.0       99812
5.0       33340
6.0       14046
7.0        7000
8.0        5264
9.0        1782
10.0        780
Name: count, dtype: int64
                              user_id      artistname  \
78   07f0fc3be95dcd878966b1f9572ff670            C418   
83   07f0fc3be95dcd878966b1f9572ff670            C418   
85   07f0fc3be95dcd878966b1f9572ff670            C418   
91   07f0fc3be95dcd878966b1f9572ff670            C418   
107  07f0fc3be95dcd878966b1f9572ff670  The Glitch Mob   
121  07f0fc3be95dcd878966b1f9572ff670            C418   
123  07f0fc3be95dcd878966b1f9572ff670          Zero 7   
125  07f0fc3be95dcd878966b1f9572ff670          Bonobo   
137  07f0fc3be95dcd878966b1f9572ff670     Johnny Cash   
139  07f0fc3be95dcd878966b1f9572ff670   Pretty Lights   

                           trackname playlistname    artist_clean  \
78            Droopy Likes Your Face         C418            c418   
83                     I Jike My Lob 

In [18]:
# def match_artist(user_artist, master_artists, threshold=85):
#     result = process.extractOne(user_artist, master_artists)
#     if result and result[1] >= threshold:
#         return result[0]
#     return None

# master_artists = df_master['artist_clean'].unique()
# df_user['artist_matched'] = df_user['artist_clean'].apply(
#     lambda x: match_artist(x, master_artists)
# )

# # Merge user data dengan audio features master via matched artist
# merged = df_user.merge(
#     df_master[['artist_clean'] + audio_features + ['primary_genre', 'popularity']],
#     left_on='artist_matched',
#     right_on='artist_clean',
#     how='left'
# )

In [14]:
def normalize(name):
    return str(name).lower().strip().replace('.', '').replace('-', ' ').replace('feat', '').replace('ft', '')

df_user['artist_key'] = df_user['artistname'].map(normalize)
df_master['artist_key'] = df_master['artist_main'].map(normalize)

In [15]:
# Buat lookup dictionary dari master (sangat ringan di memory)
master_lookup = dict(zip(df_master['artist_key'], df_master['id']))

# Proses df_user dalam chunks
CHUNK_SIZE = 500_000
results = []

for i in range(0, len(df_user), CHUNK_SIZE):
    chunk = df_user.iloc[i:i+CHUNK_SIZE].copy()
    chunk['id'] = chunk['artist_key'].map(master_lookup)
    results.append(chunk)
    print(f"Processed {min(i+CHUNK_SIZE, len(df_user)):,} / {len(df_user):,}")

df_merged = pd.concat(results, ignore_index=True)

Processed 500,000 / 8,549,544
Processed 1,000,000 / 8,549,544
Processed 1,500,000 / 8,549,544
Processed 2,000,000 / 8,549,544
Processed 2,500,000 / 8,549,544
Processed 3,000,000 / 8,549,544
Processed 3,500,000 / 8,549,544
Processed 4,000,000 / 8,549,544
Processed 4,500,000 / 8,549,544
Processed 5,000,000 / 8,549,544
Processed 5,500,000 / 8,549,544
Processed 6,000,000 / 8,549,544
Processed 6,500,000 / 8,549,544
Processed 7,000,000 / 8,549,544
Processed 7,500,000 / 8,549,544
Processed 8,000,000 / 8,549,544
Processed 8,500,000 / 8,549,544
Processed 8,549,544 / 8,549,544


In [16]:
df_merged

,user_id,artistname,trackname,playlistname,artist_clean,track_clean,play_count,artist_key,id
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elvis costello,(the angels wanna wear my) red shoes,1.0,elvis costello,64PY5B3h9mqHjyyJ9B6U2l
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elvis costello & the attractions,"(what's so funny 'bout) peace, love and unders...",1.0,elvis costello & the attractions,6ohB3BC0bgPbb23wQqBvN6
2,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elvis costello & the attractions,accidents will happen,1.0,elvis costello & the attractions,6ohB3BC0bgPbb23wQqBvN6
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elvis costello,alison,1.0,elvis costello,64PY5B3h9mqHjyyJ9B6U2l
4,9cc0cfd4d7d7885102480dd99e7a90d6,Lissie,All Be Okay,HARD ROCK 2010,lissie,all be okay,1.0,lissie,2zi6avisctlzNHw7HjK7KM
...,...,...,...,...,...,...,...,...,...
8549539,2302bf9c64dc63d88a750215ed187f2c,Mötley Crüe,Wild Side,iPhone,mötley crüe,wild side,1.0,mötley crüe,7GxiKcPz9yl4ZQ6ImHIWqB
8549540,2302bf9c64dc63d88a750215ed187f2c,John Lennon,Woman,iPhone,john lennon,woman,1.0,john lennon,5GMQdzgtI7vtpmtps2YiYx
8549541,2302bf9c64dc63d88a750215ed187f2c,Tom Petty,You Don't Know How It Feels,iPhone,tom petty,you don't know how it feels,1.0,tom petty,7HNFphOjDJpUGiseV5YvPZ
8549542,2302bf9c64dc63d88a750215ed187f2c,Tom Petty,You Wreck Me,iPhone,tom petty,you wreck me,2.0,tom petty,7HNFphOjDJpUGiseV5YvPZ


In [22]:
# df_merged.to_csv("df_merged.csv", index=False)

In [17]:
sample_merged = df_merged.sample(n=2000, random_state=42)
sample_merged.to_csv("sample_merged.csv", index=False)

In [18]:

# Load full dataset
# df = df_merged
# master = df

# ----------------------------------------------------------
# STEP 1: Drop artist_key (terpotong/salah)
# ----------------------------------------------------------
df_merged = df_merged.drop(columns=['artist_key'])

# ----------------------------------------------------------
# STEP 2: Handle duplikat user + track
# Gabung play_count kalau ada user yang dengarkan lagu sama 2x
# ----------------------------------------------------------
df_merged = df_merged.groupby(
    ['user_id','artistname','trackname','playlistname',
     'artist_clean','track_clean','id'],
    as_index=False
).agg({'play_count': 'sum'})

# ----------------------------------------------------------
# STEP 3: Flag lagu yang punya varian (remix, live, dll)
# Tetap pakai 'id' sebagai primary key item, bukan trackname
# ----------------------------------------------------------
id_variant_count = df_merged.groupby('id')['trackname'].nunique()
df_merged['has_variant'] = df_merged['id'].map(id_variant_count > 1)

# ----------------------------------------------------------
# STEP 4: Join audio features dari master
# ----------------------------------------------------------
audio_cols = ['id','valence','energy','danceability','acousticness',
              'tempo','loudness','instrumentalness','speechiness',
              'liveness','genres','popularity','year','mode','explicit']

df_merged = df_merged.merge(
    df_master[audio_cols].drop_duplicates(subset=['id']),
    on='id', how='left'
)

# Verifikasi
missing_pct = df_merged['valence'].isnull().mean() * 100
print(f"Audio features missing: {missing_pct:.1f}%")  # harusnya < 5%

# ----------------------------------------------------------
# STEP 5: Handle sparsity — pisahkan cold vs warm users
# ----------------------------------------------------------
interactions = df_merged.groupby('user_id').size()
warm_users = interactions[interactions >= 5].index
cold_users = interactions[interactions < 5].index

df_warm = df_merged[df_merged['user_id'].isin(warm_users)].copy()   # → untuk Collaborative Filtering
df_cold = df_merged[df_merged['user_id'].isin(cold_users)].copy()   # → untuk Content-Based Filtering

print(f"Warm users (CF): {len(warm_users):,}")
print(f"Cold users (CBF): {len(cold_users):,}")

# # ----------------------------------------------------------
# # Simpan hasil
# # ----------------------------------------------------------
# df.to_csv('merged_clean.csv', index=False)
# df_warm.to_csv('merged_warm.csv', index=False)
# df_cold.to_csv('merged_cold.csv', index=False)

print("Done! Dataset siap untuk modeling.")

Audio features missing: 0.0%
Warm users (CF): 14,880
Cold users (CBF): 723
Done! Dataset siap untuk modeling.


### **Collaborative Filltering** --> Df_warm

In [19]:
# --- 1A. Encode user & item ke integer ---
df_warm['user_idx'] = df_warm['user_id'].astype('category').cat.codes
df_warm['item_idx'] = df_warm['id'].astype('category').cat.codes

# Simpan mapping untuk decode balik nanti (index → id asli)
user_map = dict(enumerate(df_warm['user_id'].astype('category').cat.categories))
item_map = dict(enumerate(df_warm['id'].astype('category').cat.categories))

n_users = df_warm['user_idx'].nunique()
n_items = df_warm['item_idx'].nunique()
print(f"Matrix size: {n_users} users × {n_items} items")

# # --- 1B. Buat sparse matrix ---
# # Kenapa sparse? Karena 99.9% cell kosong — dense matrix akan makan RAM sangat besar
# # Sparse matrix hanya menyimpan nilai yang tidak nol
# sparse_matrix = csr_matrix(
#     (df_warm['play_count'].values,
#      (df_warm['user_idx'].values, df_warm['item_idx'].values)),
#     shape=(n_users, n_items)
# )

# print(f"Sparse matrix shape : {sparse_matrix.shape}")
# print(f"Non-zero entries    : {sparse_matrix.nnz:,}")
# print(f"Memory (sparse)     : {sparse_matrix.data.nbytes / 1024:.0f} KB")

Matrix size: 14880 users × 15502 items


#### **Train-Test Split**

In [20]:
def train_test_split_per_user(df, test_ratio=0.2, min_interactions=5, random_state=42):
    """
    Split per user: ambil 20% interaksi terbawah (play_count terkecil) sebagai test.
    
    Kenapa play_count terkecil untuk test?
    Karena lagu yang jarang didengar lebih sulit diprediksi — 
    ini membuat evaluasi lebih realistis (tidak terlalu mudah).
    """
    np.random.seed(random_state)
    train_data = []
    test_data  = []

    for user_id, group in df.groupby('user_idx'):
        group = group.sort_values('play_count', ascending=False)
        n = len(group)

        if n < min_interactions:
            # Terlalu sedikit — semua masuk training, skip evaluasi
            train_data.append(group)
            continue

        n_test = max(1, int(n * test_ratio))
        train_data.append(group.iloc[:-n_test])
        test_data.append(group.iloc[-n_test:])

    df_train = pd.concat(train_data).reset_index(drop=True)
    df_test  = pd.concat(test_data).reset_index(drop=True)
    return df_train, df_test

df_train, df_test = train_test_split_per_user(df_warm)
print(f"Train : {len(df_train):,} rows ({len(df_train)/len(df_warm)*100:.0f}%)")
print(f"Test  : {len(df_test):,} rows ({len(df_test)/len(df_warm)*100:.0f}%)")

# Rebuild sparse matrix dari train saja
train_matrix = csr_matrix(
    (df_train['play_count'].values,
     (df_train['user_idx'].values, df_train['item_idx'].values)),
    shape=(n_users, n_items)
)

Train : 6,843,981 rows (80%)
Test  : 1,703,611 rows (20%)


#### **Train ALS**

In [21]:
# ----------------------------------------------------------
# KUNCI: Encode SETELAH filter, bukan sebelum
# Supaya index mulai dari 0 dan konsisten dengan df_warm saja
# ----------------------------------------------------------
df_warm['user_idx'] = df_warm['user_id'].astype('category').cat.codes
df_warm['item_idx'] = df_warm['id'].astype('category').cat.codes

# Simpan mapping
user_map = dict(enumerate(df_warm['user_id'].astype('category').cat.categories))
item_map = dict(enumerate(df_warm['id'].astype('category').cat.categories))

n_users = df_warm['user_idx'].nunique()
n_items = df_warm['item_idx'].nunique()

print(f"n_users      : {n_users}")
print(f"n_items      : {n_items}")
print(f"Max user_idx : {df_warm['user_idx'].max()}  ← harus == n_users-1")
print(f"Max item_idx : {df_warm['item_idx'].max()}  ← harus == n_items-1")


train_matrix = csr_matrix(
    (df_train['play_count'].values,
     (df_train['user_idx'].values, df_train['item_idx'].values)),
    shape=(n_users, n_items)
)

print(f"n_users           : {n_users}")
print(f"n_items           : {n_items}")
print(f"Train matrix shape: {train_matrix.shape}")

n_users      : 14880
n_items      : 15502
Max user_idx : 14879  ← harus == n_users-1
Max item_idx : 15501  ← harus == n_items-1
n_users           : 14880
n_items           : 15502
Train matrix shape: (14880, 15502)


In [22]:
import implicit

als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    alpha=40,
    iterations=20,
    random_state=42
)

print("Training ALS...")
als_model.fit(train_matrix)  # ← GANTI dari sparse_matrix
print("ALS training selesai!")

user_idx_sample = 0

recommendations, scores = als_model.recommend(
    userid=user_idx_sample,
    user_items=train_matrix[user_idx_sample],  # ← GANTI dari sparse_matrix
    N=10,
    filter_already_liked_items=True
)

print(f"\nTop-10 untuk user: {user_map[user_idx_sample]}")
for rank, (item_idx, score) in enumerate(zip(recommendations, scores), 1):
    song_id   = item_map[item_idx]
    song_name = df_warm[df_warm['id'] == song_id]['trackname'].iloc[0]
    artist    = df_warm[df_warm['id'] == song_id]['artistname'].iloc[0]
    print(f"  {rank:>2}. {song_name} — {artist}  (score: {score:.4f})")

Training ALS...


  0%|          | 0/20 [00:00<?, ?it/s]

ALS training selesai!

Top-10 untuk user: 00055176fea33f6e027cd3302289378b
   1. Centuries — Fall Out Boy  (score: 1.0125)
   2. Welcome To The Black Parade — My Chemical Romance  (score: 1.0028)
   3. Me And My Broken Heart — Rixton  (score: 0.9872)
   4. Dirty Little Secret — The All-American Rejects  (score: 0.9790)
   5. Say Something — A Great Big World  (score: 0.9633)
   6. The Man Who Can't Be Moved — The Script  (score: 0.9618)
   7. Misery Business — Paramore  (score: 0.9543)
   8. Lay Me Down — Sam Smith  (score: 0.9208)
   9. I Will Never Let You Down — Rita Ora  (score: 0.9183)
  10. Cool Kids — Echosmith  (score: 0.9131)


In [29]:
# import implicit
# print(implicit.__version__)

#### **Train SVD**

In [23]:
from scipy.sparse.linalg import svds

In [24]:
# ----------------------------------------------------------
# Train SVD pakai scipy — tidak perlu library tambahan
# ----------------------------------------------------------

# Normalisasi play_count ke log-scale dulu
from scipy.sparse import csr_matrix

# Buat matrix dari df_train (bukan full sparse_matrix)
train_matrix = csr_matrix(
    (np.log1p(df_train['play_count'].values),
     (df_train['user_idx'].values, df_train['item_idx'].values)),
    shape=(n_users, n_items)
)

# SVD decomposition
# k = jumlah latent factors, sama dengan ALS (64)
k = 64
U, sigma, Vt = svds(train_matrix, k=k)

# Konversi sigma ke diagonal matrix
sigma_diag = np.diag(sigma)

# Rekonstruksi predicted scores untuk semua user × item
# Shape: (n_users, n_items)
predicted_scores = np.dot(np.dot(U, sigma_diag), Vt)

print(f"U shape     : {U.shape}")        # (n_users, k)
print(f"sigma shape : {sigma_diag.shape}")  # (k, k)
print(f"Vt shape    : {Vt.shape}")       # (k, n_items)
print(f"Predicted scores shape: {predicted_scores.shape}")  # (n_users, n_items)

# ----------------------------------------------------------
# Validasi — Top-10 untuk user sample
# ----------------------------------------------------------
user_idx_sample = 0
seen_items = set(df_train[df_train['user_idx'] == user_idx_sample]['item_idx'])

# Ambil scores untuk user ini, set seen items ke -inf supaya tidak muncul
user_scores = predicted_scores[user_idx_sample].copy()
for idx in seen_items:
    user_scores[idx] = -np.inf

top10_idx = np.argsort(user_scores)[::-1][:10]

print(f"\nTop-10 SVD untuk user: {user_map[user_idx_sample]}")
for rank, item_idx in enumerate(top10_idx, 1):
    song_id   = item_map[item_idx]
    song_name = df_warm[df_warm['id'] == song_id]['trackname'].iloc[0]
    artist    = df_warm[df_warm['id'] == song_id]['artistname'].iloc[0]
    score     = user_scores[item_idx]
    print(f"  {rank:>2}. {song_name} — {artist}  (score: {score:.4f})")

U shape     : (14880, 64)
sigma shape : (64, 64)
Vt shape    : (64, 15502)
Predicted scores shape: (14880, 15502)

Top-10 SVD untuk user: 00055176fea33f6e027cd3302289378b
   1. Centuries — Fall Out Boy  (score: 1.3685)
   2. The Baddest Man Alive — The Black Keys  (score: 0.7420)
   3. Welcome To The Black Parade — My Chemical Romance  (score: 0.6601)
   4. Misery Business — Paramore  (score: 0.5578)
   5. America (Glee Cast Version) — Glee Cast  (score: 0.5024)
   6. I Am the Best (내가 제일 잘 나가) — 2NE1  (score: 0.4653)
   7. In Too Deep — Sum 41  (score: 0.4315)
   8. Human Race — Three Days Grace  (score: 0.3995)
   9. Since U Been Gone — A Day To Remember  (score: 0.3949)
  10. Nation Of Wusses — Infected Mushroom  (score: 0.3698)


#### **Evaluasi**

In [25]:
K = 10

# ----------------------------------------------------------
# Fungsi evaluasi — sama untuk kedua model
# ----------------------------------------------------------
def evaluate(recommendations, df_test, K):
    """
    recommendations: dict {user_idx: [list item_idx terurut]}
    df_test        : dataframe test set
    """
    precisions, recalls, ndcgs = [], [], []

    test_items = df_test.groupby('user_idx')['item_idx'].apply(set).to_dict()

    for user_idx, rec_list in recommendations.items():
        actual = test_items.get(user_idx, set())
        if not actual:
            continue

        # Precision@K
        hits = [1 if item in actual else 0 for item in rec_list[:K]]
        precision = sum(hits) / K

        # Recall@K
        recall = sum(hits) / len(actual)

        # NDCG@K — lagu relevan di posisi atas dapat bobot lebih tinggi
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(actual), K)))
        ndcg = dcg / idcg if idcg > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f'Precision@{K}': round(np.mean(precisions), 4),
        f'Recall@{K}'   : round(np.mean(recalls), 4),
        f'NDCG@{K}'     : round(np.mean(ndcgs), 4),
        'n_users_evaluated': len(precisions)
    }

# ----------------------------------------------------------
# Generate rekomendasi ALS untuk semua test users
# ----------------------------------------------------------
print("Generating ALS recommendations...")
test_users = df_test['user_idx'].unique()
recs_als = {}

for user_idx in test_users:
    rec_items, _ = als_model.recommend(
        userid=user_idx,
        user_items=train_matrix[user_idx],
        N=K,
        filter_already_liked_items=True
    )
    recs_als[user_idx] = list(rec_items)

# ----------------------------------------------------------
# Generate rekomendasi SVD untuk semua test users
# ----------------------------------------------------------
print("Generating SVD recommendations...")
user_seen = df_train.groupby('user_idx')['item_idx'].apply(set).to_dict()
recs_svd = {}

for user_idx in test_users:
    scores     = predicted_scores[user_idx].copy()
    seen       = user_seen.get(user_idx, set())
    for idx in seen:
        scores[idx] = -np.inf
    top_k = np.argsort(scores)[::-1][:K]
    recs_svd[user_idx] = list(top_k)

# ----------------------------------------------------------
# Hitung metrics
# ----------------------------------------------------------
print("Evaluating...")
metrics_als = evaluate(recs_als, df_test, K)
metrics_svd = evaluate(recs_svd, df_test, K)

# ----------------------------------------------------------
# Tampilkan perbandingan
# ----------------------------------------------------------
print("\n" + "=" * 55)
print(f"{'Metric':<20} {'ALS':>10} {'SVD':>10} {'Winner':>12}")
print("=" * 55)

for metric in [f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']:
    als_val = metrics_als[metric]
    svd_val = metrics_svd[metric]
    winner  = 'ALS ✓' if als_val >= svd_val else 'SVD ✓'
    print(f"{metric:<20} {als_val:>10.4f} {svd_val:>10.4f} {winner:>12}")

print("=" * 55)
print(f"{'Users evaluated':<20} {metrics_als['n_users_evaluated']:>10} {metrics_svd['n_users_evaluated']:>10}")

Generating ALS recommendations...
Generating SVD recommendations...
Evaluating...

Metric                      ALS        SVD       Winner
Precision@10             0.0759     0.0638        ALS ✓
Recall@10                0.0410     0.0254        ALS ✓
NDCG@10                  0.0859     0.0715        ALS ✓
Users evaluated           14880      14880


#### **Hyperparameter Tuning --> ALS**

In [26]:
import itertools

# ----------------------------------------------------------
# Grid Search ALS — versi cepat (12 kombinasi)
# ----------------------------------------------------------
param_grid = {
    'factors'       : [32, 64, 128, 256],
    'alpha'         : [10, 40, 80],
    'regularization': [0.01, 0.05, 0.1],
    'iterations'    : [20]
}

keys   = list(param_grid.keys())
values = list(param_grid.values())
combos = list(itertools.product(*values))
print(f"Total kombinasi : {len(combos)}")

results = []

for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    
    model = implicit.als.AlternatingLeastSquares(
        factors        = params['factors'],
        regularization = params['regularization'],
        alpha          = params['alpha'],
        iterations     = params['iterations'],
        random_state   = 42
    )
    model.fit(train_matrix)
    
    recs = {}
    for user_idx in df_test['user_idx'].unique():
        rec_items, _ = model.recommend(
            userid     = user_idx,
            user_items = train_matrix[user_idx],
            N          = 10,
            filter_already_liked_items=True
        )
        recs[user_idx] = list(rec_items)
    
    metrics = evaluate(recs, df_test, K=10)
    results.append({**params, **metrics})
    print(f"  [{i+1}/{len(combos)}] factors={params['factors']}, "
          f"alpha={params['alpha']} → NDCG={metrics['NDCG@10']:.4f}")

# ----------------------------------------------------------
# Hasil
# ----------------------------------------------------------
df_results = pd.DataFrame(results).sort_values('NDCG@10', ascending=False)

print("\n" + "=" * 75)
print(f"{'factors':>8} {'alpha':>6} {'reg':>6} {'iter':>5} │ "
      f"{'Precision':>10} {'Recall':>10} {'NDCG':>10}")
print("=" * 75)

for _, row in df_results.iterrows():
    marker = " ← best" if _ == df_results.index[0] else ""
    print(f"{int(row['factors']):>8} {int(row['alpha']):>6} "
          f"{row['regularization']:>6} {int(row['iterations']):>5} │ "
          f"{row['Precision@10']:>10.4f} {row['Recall@10']:>10.4f} "
          f"{row['NDCG@10']:>10.4f}{marker}")

print("=" * 75)
print(f"\nBaseline → Precision: 0.1201 | Recall: 0.0313 | NDCG: 0.1241")

best = df_results.iloc[0]
print(f"\nBest params:")
print(f"  factors        : {int(best['factors'])}")
print(f"  alpha          : {int(best['alpha'])}")
print(f"  regularization : {best['regularization']}")
print(f"  NDCG@10        : {best['NDCG@10']:.4f}")

Total kombinasi : 36


  0%|          | 0/20 [00:00<?, ?it/s]

  [1/36] factors=32, alpha=10 → NDCG=0.1108


  0%|          | 0/20 [00:00<?, ?it/s]

  [2/36] factors=32, alpha=10 → NDCG=0.1115


...

  [34/36] factors=256, alpha=80 → NDCG=0.0693


  0%|          | 0/20 [00:00<?, ?it/s]

  [35/36] factors=256, alpha=80 → NDCG=0.0695


  0%|          | 0/20 [00:00<?, ?it/s]

  [36/36] factors=256, alpha=80 → NDCG=0.0705

 factors  alpha    reg  iter │  Precision     Recall       NDCG
      32     10    0.1    20 │     0.0986     0.0446     0.1116 ← best
      32     10   0.05    20 │     0.0984     0.0447     0.1115
      32     10   0.01    20 │     0.0983     0.0445     0.1108
      64     10    0.1    20 │     0.0922     0.0464     0.1052
      64     10   0.05    20 │     0.0923     0.0463     0.1051
      64     10   0.01    20 │     0.0908     0.0461     0.1038
     128     10    0.1    20 │     0.0845     0.0462     0.0978
     128     10   0.05    20 │     0.0844     0.0462     0.0974
     128     10   0.01    20 │     0.0835     0.0458     0.0965
      32     40   0.05    20 │     0.0847     0.0416     0.0946
      32     40    0.1    20 │     0.0847     0.0415     0.0944
      32     40   0.01    20 │     0.0842     0.0416     0.0942
      64     40    0.1    20 │     0.0807     0.0431     0.0917
      64     40   0.05    20 │     0.0802     0.04

In [27]:
print("Tuning iterations dengan best params...")
results_iter = []

for itr in [20, 30, 50, 100]:
    model = implicit.als.AlternatingLeastSquares(
        factors        = 32,
        alpha          = 10,
        regularization = 0.1,
        iterations     = itr,
        random_state   = 42
    )
    model.fit(train_matrix)
    
    recs = {}
    for user_idx in df_test['user_idx'].unique():
        rec_items, _ = model.recommend(
            userid     = user_idx,
            user_items = train_matrix[user_idx],
            N          = 10,
            filter_already_liked_items=True
        )
        recs[user_idx] = list(rec_items)
    
    metrics = evaluate(recs, df_test, K=10)
    results_iter.append({'iterations': itr, **metrics})
    print(f"  iterations={itr:>3} → NDCG={metrics['NDCG@10']:.4f}")

# Hasil
print("\n" + "=" * 55)
print(f"{'iterations':>10} │ {'Precision':>10} {'Recall':>10} {'NDCG':>10}")
print("=" * 55)
for r in results_iter:
    print(f"{r['iterations']:>10} │ {r['Precision@10']:>10.4f} "
          f"{r['Recall@10']:>10.4f} {r['NDCG@10']:>10.4f}")
print("=" * 55)

Tuning iterations dengan best params...


  0%|          | 0/20 [00:00<?, ?it/s]

  iterations= 20 → NDCG=0.1116


  0%|          | 0/30 [00:00<?, ?it/s]

  iterations= 30 → NDCG=0.1115


  0%|          | 0/50 [00:00<?, ?it/s]

  iterations= 50 → NDCG=0.1107


  0%|          | 0/100 [00:00<?, ?it/s]

  iterations=100 → NDCG=0.1101

iterations │  Precision     Recall       NDCG
        20 │     0.0986     0.0446     0.1116
        30 │     0.0988     0.0452     0.1115
        50 │     0.0979     0.0450     0.1107
       100 │     0.0975     0.0446     0.1101


#### **Final Best Parameter ALS**

In [29]:
als_final = implicit.als.AlternatingLeastSquares(
    factors        = 32,
    alpha          = 10,
    regularization = 0.1,
    iterations     = 20,
    random_state   = 42
)

als_final.fit(train_matrix)

# Evaluasi final
recs_final = {}
for user_idx in df_test['user_idx'].unique():
    rec_items, _ = als_final.recommend(
        userid     = user_idx,
        user_items = train_matrix[user_idx],
        N          = 10,
        filter_already_liked_items=True
    )
    recs_final[user_idx] = list(rec_items)

metrics_final = evaluate(recs_final, df_test, K=10)

baseline_ndcg = 0.0859

print("=" * 55)
print("RINGKASAN PERJALANAN TUNING ALS")
print("=" * 55)
print(f"{'':25} {'Precision':>10} {'Recall':>10} {'NDCG':>10}")
print("-" * 55)
print(f"{'Baseline':25} {'0.0759':>10} {'0.0410':>10} {'0.0859':>10}")
print(f"{'Setelah tune factors':25} {'0.0986':>10} {'0.0446':>10} {'0.1116':>10}")
print(f"{'Final (f=64,a=10,i=20)':25} "
      f"{metrics_final['Precision@10']:>10.4f} "
      f"{metrics_final['Recall@10']:>10.4f} "
      f"{metrics_final['NDCG@10']:>10.4f}")
print("=" * 55)
print(f"\nTotal improvement NDCG : "
      f"+{(metrics_final['NDCG@10'] - baseline_ndcg)/baseline_ndcg*100:.0f}% dari baseline")

  0%|          | 0/20 [00:00<?, ?it/s]

RINGKASAN PERJALANAN TUNING ALS
                           Precision     Recall       NDCG
-------------------------------------------------------
Baseline                      0.0759     0.0410     0.0859
Setelah tune factors          0.0986     0.0446     0.1116
Final (f=64,a=10,i=20)        0.0986     0.0446     0.1116

Total improvement NDCG : +30% dari baseline


### **CONTENT-BASED FILTERING** --> DF_cold

In [30]:
# Cek dulu features yang ada dan kondisinya

audio_features = ['valence','energy','danceability','acousticness',
                  'instrumentalness','speechiness','liveness',
                  'loudness','tempo','popularity']

print("Feature stats:")
print(df_cold[audio_features].describe().round(3))

print(f"\nMissing values:")
print(df_cold[audio_features].isnull().sum())

print(f"\nUnique lagu di cold users: {df_cold['id'].nunique()}")
print(f"Total cold users         : {df_cold['user_id'].nunique()}")

Feature stats:
        valence    energy  danceability  acousticness  instrumentalness  \
count  1442.000  1442.000      1442.000      1442.000          1442.000   
mean      0.474     0.642         0.565         0.272             0.109   
std       0.241     0.238         0.166         0.304             0.259   
min       0.000     0.000         0.000         0.000             0.000   
25%       0.273     0.489         0.452         0.013             0.000   
50%       0.466     0.693         0.588         0.132             0.000   
75%       0.670     0.842         0.684         0.492             0.015   
max       0.975     0.994         0.946         0.995             1.000   

       speechiness  liveness  loudness     tempo  popularity  
count     1442.000  1442.000  1442.000  1442.000    1442.000  
mean         0.074     0.213    -7.989   121.211      49.447  
std          0.071     0.191     4.633    28.534      13.459  
min          0.000     0.024   -35.651     0.000       0.

In [31]:
from sklearn.preprocessing import MinMaxScaler

item_profiles = df_cold[['id'] + audio_features].drop_duplicates(subset='id').copy()
item_profiles = item_profiles.set_index('id')

scaler = MinMaxScaler()
item_profiles[audio_features] = scaler.fit_transform(item_profiles[audio_features])

print(f"Item profile matrix shape: {item_profiles.shape}")
print(f"\nSetelah normalisasi — semua nilai harus 0.0 - 1.0:")
print(item_profiles.describe().round(3))

Item profile matrix shape: (846, 10)

Setelah normalisasi — semua nilai harus 0.0 - 1.0:
       valence   energy  danceability  acousticness  instrumentalness  \
count  846.000  846.000       846.000       846.000           846.000   
mean     0.488    0.635         0.586         0.283             0.119   
std      0.253    0.245         0.178         0.312             0.266   
min      0.000    0.000         0.000         0.000             0.000   
25%      0.282    0.469         0.467         0.014             0.000   
50%      0.482    0.683         0.603         0.140             0.000   
75%      0.690    0.846         0.711         0.494             0.028   
max      1.000    1.000         1.000         1.000             1.000   

       speechiness  liveness  loudness    tempo  popularity  
count      846.000   846.000   846.000  846.000     846.000  
mean         0.167     0.187     0.799    0.584       0.603  
std          0.159     0.189     0.138    0.139       0.174  
min  

In [32]:
# ----------------------------------------------------------
# Definisi fungsi — jalankan cell ini dulu
# ----------------------------------------------------------
def build_user_profile(user_id, df_user, item_profiles, audio_features):
    user_songs = df_user[df_user['user_id'] == user_id][['id','play_count']]
    user_songs = user_songs[user_songs['id'].isin(item_profiles.index)]
    
    if len(user_songs) == 0:
        return None
    
    profiles = item_profiles.loc[user_songs['id'], audio_features]
    weights  = user_songs.set_index('id')['play_count']
    
    user_profile = np.average(profiles, weights=weights, axis=0)
    return user_profile

# Step 2: Build user profile — jalankan dulu
sample_user = df_cold['user_id'].iloc[0]
profile = build_user_profile(sample_user, df_cold, item_profiles, audio_features)

print(f"User profile untuk: {sample_user}")
print(pd.Series(profile, index=audio_features).round(3))

User profile untuk: 00152c870313100559aad7b097d9c1f5
valence             0.711
energy              0.702
danceability        0.833
acousticness        0.087
instrumentalness    0.000
speechiness         0.606
liveness            0.099
loudness            0.895
tempo               0.432
popularity          0.487
dtype: float64


In [34]:
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------------------------------------
# Definisi fungsi recommend_cbf — jalankan dulu
# ----------------------------------------------------------
def recommend_cbf(user_id, df_user, item_profiles, audio_features, N=10):
    user_profile = build_user_profile(user_id, df_user, item_profiles, audio_features)
    if user_profile is None:
        return []
    
    seen_ids   = set(df_user[df_user['user_id'] == user_id]['id'])
    candidates = item_profiles[~item_profiles.index.isin(seen_ids)]
    
    similarities = cosine_similarity(
        user_profile.reshape(1, -1),
        candidates[audio_features].values
    )[0]
    
    top_idx    = np.argsort(similarities)[::-1][:N]
    top_ids    = candidates.index[top_idx]
    top_scores = similarities[top_idx]
    
    return list(zip(top_ids, top_scores))

# ----------------------------------------------------------
# Validasi — Top-10 untuk sample user
# ----------------------------------------------------------
sample_user    = df_cold['user_id'].iloc[0]
recommendations = recommend_cbf(sample_user, df_cold, item_profiles, audio_features, N=10)

print(f"Lagu yang sudah didengar user ini:")
seen = df_cold[df_cold['user_id'] == sample_user][['trackname','artistname','play_count']]
print(seen.to_string(index=False))

print(f"\nTop-10 Rekomendasi CBF:")
for rank, (song_id, score) in enumerate(recommendations, 1):
    row = df_cold[df_cold['id'] == song_id].iloc[0]
    print(f"  {rank:>2}. {row['trackname']} — {row['artistname']}  (similarity: {score:.4f})")

Lagu yang sudah didengar user ini:
                                     trackname artistname  play_count
The Militia - Feat. Big Shug And Freddie Foxxx Gang Starr         1.0

Top-10 Rekomendasi CBF:
   1. Watcha Gonna Do (feat. Timbaland) — Missy Elliott  (similarity: 0.9927)
   2. Don't Let It Go To Your Head — Brand Nubian  (similarity: 0.9896)
   3. The Conversation — Ky-Mani Marley  (similarity: 0.9888)
   4. Regulate — Warren G  (similarity: 0.9871)
   5. Lose My Mind — Young Buck  (similarity: 0.9852)
   6. Can't Hold Us - feat. Ray Dalton — Macklemore & Ryan Lewis  (similarity: 0.9851)
   7. Lighters — Bad Meets Evil  (similarity: 0.9814)
   8. Sexual Eruption - Fyre Dept. Remix — Snoop Dogg  (similarity: 0.9812)
   9. Babylon (feat. Kendrick Lamar) — SZA  (similarity: 0.9811)
  10. All That Matters — Justin Bieber  (similarity: 0.9809)


In [35]:
# ----------------------------------------------------------
# Definisi fungsi split & evaluasi — jalankan dulu
# ----------------------------------------------------------
def train_test_split_cold(df, test_ratio=0.2, random_state=42):
    np.random.seed(random_state)
    train_data, test_data = [], []
    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('play_count', ascending=False)
        n     = len(group)
        n_test = max(1, int(n * test_ratio))
        train_data.append(group.iloc[:-n_test])
        test_data.append(group.iloc[-n_test:])
    return pd.concat(train_data).reset_index(drop=True), \
           pd.concat(test_data).reset_index(drop=True)

def evaluate_cbf(recommendations, df_test, K=10):
    precisions, recalls, ndcgs = [], [], []
    test_items = df_test.groupby('user_id')['id'].apply(set).to_dict()

    for user_id, rec_list in recommendations.items():
        actual = test_items.get(user_id, set())
        if not actual:
            continue

        hits      = [1 if item in actual else 0 for item in rec_list[:K]]
        precision = sum(hits) / K
        recall    = sum(hits) / len(actual)

        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(actual), K)))
        ndcg = dcg / idcg if idcg > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f'Precision@{K}': round(np.mean(precisions), 4),
        f'Recall@{K}'   : round(np.mean(recalls), 4),
        f'NDCG@{K}'     : round(np.mean(ndcgs), 4),
        'n_users_evaluated': len(precisions)
    }

# ----------------------------------------------------------
# Split & generate rekomendasi untuk semua cold users
# ----------------------------------------------------------
df_cold_train, df_cold_test = train_test_split_cold(df_cold)

print(f"Cold train : {len(df_cold_train):,} rows")
print(f"Cold test  : {len(df_cold_test):,} rows")

print("\nGenerating CBF recommendations...")
cold_users = df_cold['user_id'].unique()
recs_cbf   = {}

for user_id in cold_users:
    recs = recommend_cbf(user_id, df_cold_train, item_profiles, audio_features, N=10)
    recs_cbf[user_id] = [song_id for song_id, _ in recs]

# ----------------------------------------------------------
# Evaluasi
# ----------------------------------------------------------
metrics_cbf = evaluate_cbf(recs_cbf, df_cold_test, K=10)

print("\n" + "=" * 45)
print("HASIL EVALUASI CBF (Cold Users)")
print("=" * 45)
for metric, val in metrics_cbf.items():
    print(f"{metric:<25} : {val}")

Cold train : 719 rows
Cold test  : 723 rows

Generating CBF recommendations...

HASIL EVALUASI CBF (Cold Users)
Precision@10              : 0.0015
Recall@10                 : 0.0152
NDCG@10                   : 0.0066
n_users_evaluated         : 723


#### **CBF Improvement**

In [36]:
def get_popular_items(df, N=10, exclude=set()):
    """
    Popularity berdasarkan jumlah unique users yang mendengar
    (bukan total play_count) — lebih representatif.
    Score dinormalisasi ke 0-1.
    """
    popular = (df.groupby('id')['user_id']
                 .nunique()
                 .reset_index()
                 .rename(columns={'user_id': 'listener_count'})
                 .sort_values('listener_count', ascending=False))

    popular = popular[~popular['id'].isin(exclude)]

    max_count = popular['listener_count'].max()
    popular['score'] = popular['listener_count'] / max_count

    return [(row['id'], round(row['score'], 4))
            for _, row in popular.head(N).iterrows()]

# Validasi — cek top-10 popularity baru
df_all = pd.concat([df_warm, df_cold], ignore_index=True)

# Buat lookup lokal sementara  ← baris yang kurang
_lookup = df_all.drop_duplicates('id').set_index('id')[['trackname','artistname']].to_dict('index')

print("Top-10 popularity (unique listeners):")
for rank, (song_id, score) in enumerate(get_popular_items(df_all, N=10), 1):
    info = _lookup.get(song_id, {})
    print(f"  {rank:>2}. {info.get('trackname','?'):<40} "
          f"{info.get('artistname','?'):<25} (score: {score:.4f})")

Top-10 popularity (unique listeners):
   1. Atlas - From “The Hunger Games: Catching Fire” Soundtrack Coldplay                  (score: 1.0000)
   2. Around The World                         Daft Punk                 (score: 0.9970)
   3. FourFiveSeconds                          Rihanna                   (score: 0.8809)
   4. Dangerous (feat. Sam Martin)             David Guetta              (score: 0.8312)
   5. Blame                                    Calvin Harris             (score: 0.8034)
   6. Irreplaceable                            Beyoncé                   (score: 0.7942)
   7. Birthday                                 Katy Perry                (score: 0.7731)
   8. Addicted To You - Avicii By Avicii       Avicii                    (score: 0.7615)
   9. You Are Not Alone                        Michael Jackson           (score: 0.7481)
  10. Blame Game                               Kanye West                (score: 0.7421)


In [37]:
def recommend_cold_start(user_id, df_user, item_profiles,
                          audio_features, df_all, N=10):
    """
    Hybrid recommendation untuk cold users:
    - 0 lagu  → pure popularity
    - 1 lagu  → 50% CBF + 50% popularity
    - 2+ lagu → pure CBF
    """
    user_songs = df_user[df_user['user_id'] == user_id]
    n_songs    = len(user_songs)
    seen_ids   = set(user_songs['id'])

    if n_songs == 0:
        # Tidak ada data sama sekali → pure popularity
        return get_popular_items(df_all, N=N, exclude=seen_ids)

    elif n_songs == 1:
        # 1 lagu → 50% CBF + 50% popularity
        n_cbf = N // 2       # 5 dari CBF
        n_pop = N - n_cbf    # 5 dari popularity

        cbf_recs  = recommend_cbf(user_id, df_user, item_profiles,
                                   audio_features, N=n_cbf)
        cbf_ids   = set(r[0] for r in cbf_recs)

        pop_recs  = get_popular_items(df_all, N=n_pop,
                                       exclude=seen_ids | cbf_ids)

        return cbf_recs + pop_recs

    else:
        # 2-4 lagu → pure CBF
        return recommend_cbf(user_id, df_user, item_profiles,
                              audio_features, N=N)

# ----------------------------------------------------------
# Validasi — bandingkan user 1 lagu vs user 2+ lagu
# ----------------------------------------------------------
interactions_cold = df_cold.groupby('user_id').size()
user_1_song  = interactions_cold[interactions_cold == 1].index[0]
user_2_songs = interactions_cold[interactions_cold >= 2].index[0]

for user_id, label in [(user_1_song, '1 lagu'), (user_2_songs, '2+ lagu')]:
    seen = df_cold[df_cold['user_id'] == user_id][['trackname','artistname']]
    print(f"\n{'='*55}")
    print(f"User ({label}): {user_id}")
    print(f"Lagu yang didengar:")
    print(seen.to_string(index=False))
    
    recs = recommend_cold_start(user_id, df_cold, item_profiles,
                                 audio_features, df_cold, N=10)
    print(f"\nRekomendasi hybrid:")
    for rank, (song_id, score) in enumerate(recs, 1):
        row = df_cold[df_cold['id'] == song_id].iloc[0]
        src = 'CBF' if rank <= 5 else 'POP'
        label_src = f'[{src}]' if interactions_cold[user_id] == 1 else '[CBF]'
        print(f"  {rank:>2}. {label_src} {row['trackname']} — {row['artistname']}")


User (1 lagu): 00152c870313100559aad7b097d9c1f5
Lagu yang didengar:
                                     trackname artistname
The Militia - Feat. Big Shug And Freddie Foxxx Gang Starr

Rekomendasi hybrid:
   1. [CBF] Watcha Gonna Do (feat. Timbaland) — Missy Elliott
   2. [CBF] Don't Let It Go To Your Head — Brand Nubian
   3. [CBF] The Conversation — Ky-Mani Marley
   4. [CBF] Regulate — Warren G
   5. [CBF] Lose My Mind — Young Buck
   6. [POP] Photograph — Ed Sheeran
   7. [POP] Sexy Bitch (feat. Akon) - Featuring Akon;explicit — David Guetta
   8. [POP] Instant Crush — Daft Punk
   9. [POP] Drunk in Love — Beyoncé
  10. [POP] Jump — Rihanna

User (2+ lagu): 0077392401062a0eb2bef08c4ded8bb5
Lagu yang didengar:
                                trackname                      artistname
California Love - Original Mix (Explicit)                            2Pac
                  Egyptian March, Op. 335 Wiener Johann Strauss Orchester

Rekomendasi hybrid:
   1. [CBF] Rockin' All Over The 

In [38]:
# ----------------------------------------------------------
# Split — user 1 lagu semua masuk train, skip evaluasi
# ----------------------------------------------------------
df_cold_train, df_cold_test = train_test_split_cold(df_cold)

print(f"Cold train : {len(df_cold_train):,} rows")
print(f"Cold test  : {len(df_cold_test):,} rows")
print(f"Users yang dievaluasi : {df_cold_test['user_id'].nunique()}")

# ----------------------------------------------------------
# Generate rekomendasi hybrid untuk semua cold users
# ----------------------------------------------------------
print("\nGenerating hybrid recommendations...")
recs_hybrid = {}

for user_id in df_cold['user_id'].unique():
    recs = recommend_cold_start(
        user_id, df_cold_train, item_profiles,
        audio_features, df_cold_train, N=10
    )
    recs_hybrid[user_id] = [song_id for song_id, _ in recs]

# ----------------------------------------------------------
# Evaluasi
# ----------------------------------------------------------
metrics_hybrid = evaluate_cbf(recs_hybrid, df_cold_test, K=10)

print("\n" + "=" * 50)
print(f"{'Metric':<25} {'CBF lama':>10} {'Hybrid':>10} {'Delta':>10}")
print("=" * 50)

metrics_old = {
    'Precision@10': 0.0017,
    'Recall@10'   : 0.0166,
    'NDCG@10'     : 0.0070
}

for metric in ['Precision@10', 'Recall@10', 'NDCG@10']:
    old = metrics_old[metric]
    new = metrics_hybrid[metric]
    delta = f"+{new-old:.4f}" if new >= old else f"{new-old:.4f}"
    print(f"{metric:<25} {old:>10.4f} {new:>10.4f} {delta:>10}")

print("=" * 50)
print(f"{'Users evaluated':<25} {'723':>10} {metrics_hybrid['n_users_evaluated']:>10}")

Cold train : 719 rows
Cold test  : 723 rows
Users yang dievaluasi : 723

Generating hybrid recommendations...

Metric                      CBF lama     Hybrid      Delta
Precision@10                  0.0017     0.0043    +0.0026
Recall@10                     0.0166     0.0429    +0.0263
NDCG@10                       0.0070     0.0201    +0.0131
Users evaluated                  723        723


### **Wrap Function**

In [39]:
# Checklist — jalankan ini dulu
print("Checklist objek yang dibutuhkan:")
print(f"  als_final        : {'✅' if 'als_final' in dir() else '❌'}")
print(f"  train_matrix     : {'✅' if 'train_matrix' in dir() else '❌'}")
print(f"  user_map         : {'✅' if 'user_map' in dir() else '❌'}")
print(f"  item_map         : {'✅' if 'item_map' in dir() else '❌'}")
print(f"  df_warm          : {'✅' if 'df_warm' in dir() else '❌'}")
print(f"  df_cold          : {'✅' if 'df_cold' in dir() else '❌'}")
print(f"  item_profiles    : {'✅' if 'item_profiles' in dir() else '❌'}")
print(f"  audio_features   : {'✅' if 'audio_features' in dir() else '❌'}")
print(f"  recommend_cbf    : {'✅' if 'recommend_cbf' in dir() else '❌'}")
print(f"  get_popular_items: {'✅' if 'get_popular_items' in dir() else '❌'}")

Checklist objek yang dibutuhkan:
  als_final        : ✅
  train_matrix     : ✅
  user_map         : ✅
  item_map         : ✅
  df_warm          : ✅
  df_cold          : ✅
  item_profiles    : ✅
  audio_features   : ✅
  recommend_cbf    : ✅
  get_popular_items: ✅


In [40]:
# ----------------------------------------------------------
# Reverse mapping: user_id (string) → user_idx (integer)
# Dibutuhkan supaya fungsi bisa menerima user_id asli
# bukan index integer
# ----------------------------------------------------------
user_id_to_idx = {v: k for k, v in user_map.items()}
item_id_to_name = df_warm.drop_duplicates('id').set_index('id')[['trackname','artistname']].to_dict('index')
item_id_to_name.update(
    df_cold.drop_duplicates('id').set_index('id')[['trackname','artistname']].to_dict('index')
)

# ----------------------------------------------------------
# Fungsi utama
# ----------------------------------------------------------
def get_recommendation(user_id, N=10):
    """
    Unified recommendation function.
    Otomatis pilih model berdasarkan jumlah interaksi user.

    Parameters:
        user_id : string — user_id asli dari dataset
        N       : int    — jumlah rekomendasi (default 10)

    Returns:
        list of dict: [{rank, song_id, trackname, artistname, score, source}]
    """

    # Gabungkan semua interaksi user (warm + cold)
    df_all = pd.concat([df_warm, df_cold], ignore_index=True)
    user_interactions = df_all[df_all['user_id'] == user_id]
    n_interactions    = len(user_interactions)

    # ----------------------------------------------------------
    # Routing logic
    # ----------------------------------------------------------
    if user_id not in user_id_to_idx and n_interactions == 0:
        # User benar-benar baru — tidak ada di dataset sama sekali
        source = 'popularity'
        recs   = get_popular_items(df_all, N=N)
        recs   = [(song_id, score) for song_id, score in recs]

    elif n_interactions >= 5 and user_id in user_id_to_idx:
        # Warm user → ALS
        source    = 'ALS'
        user_idx  = user_id_to_idx[user_id]
        rec_items, scores = als_final.recommend(
            userid     = user_idx,
            user_items = train_matrix[user_idx],
            N          = N,
            filter_already_liked_items=True
        )
        recs = [(item_map[item_idx], float(score))
                for item_idx, score in zip(rec_items, scores)]

    else:
        # Cold user → Hybrid CBF + Popularity
        source = 'hybrid_CBF'
        raw    = recommend_cold_start(
            user_id, user_interactions, item_profiles,
            audio_features, df_all, N=N
        )
        recs = [(song_id, float(score)) for song_id, score in raw]

    # ----------------------------------------------------------
    # Format output
    # ----------------------------------------------------------
    output = []
    for rank, (song_id, score) in enumerate(recs, 1):
        info = item_id_to_name.get(song_id, {})
        output.append({
            'rank'      : rank,
            'song_id'   : song_id,
            'trackname' : info.get('trackname', 'Unknown'),
            'artistname': info.get('artistname', 'Unknown'),
            'score'     : round(score, 4),
            'source'    : source
        })

    return output


# ----------------------------------------------------------
# Fungsi display — supaya output rapi
# ----------------------------------------------------------
def show_recommendation(user_id, N=10):
    recs = get_recommendation(user_id, N)

    if not recs:
        print(f"Tidak ada rekomendasi untuk user: {user_id}")
        return

    source = recs[0]['source']
    df_all = pd.concat([df_warm, df_cold], ignore_index=True)
    n      = len(df_all[df_all['user_id'] == user_id])

    print(f"User    : {user_id}")
    print(f"Model   : {source}  |  Interaksi: {n} lagu")
    print(f"{'─'*65}")
    print(f"{'#':>3}  {'Track':<35} {'Artist':<20} {'Score':>7}")
    print(f"{'─'*65}")
    for r in recs:
        track  = r['trackname'][:33] + '..' if len(r['trackname']) > 35 else r['trackname']
        artist = r['artistname'][:18] + '..' if len(r['artistname']) > 20 else r['artistname']
        print(f"{r['rank']:>3}. {track:<35} {artist:<20} {r['score']:>7.4f}")
    print(f"{'─'*65}")

#### **Validasi 3 Tipe User**

In [41]:
df_all = pd.concat([df_warm, df_cold], ignore_index=True)

# Ambil sample untuk tiap tipe
warm_sample  = df_warm['user_id'].iloc[0]
cold_sample  = df_cold['user_id'].iloc[0]
new_user     = 'user_baru_123'  # tidak ada di dataset

print("=" * 65)
print("TEST 1 — Warm user (≥5 interaksi) → ALS")
print("=" * 65)
show_recommendation(warm_sample)

print("\n" + "=" * 65)
print("TEST 2 — Cold user (<5 interaksi) → Hybrid CBF")
print("=" * 65)
show_recommendation(cold_sample)

print("\n" + "=" * 65)
print("TEST 3 — User baru (tidak ada di dataset) → Popularity")
print("=" * 65)
show_recommendation(new_user)

TEST 1 — Warm user (≥5 interaksi) → ALS
User    : 00055176fea33f6e027cd3302289378b
Model   : ALS  |  Interaksi: 98 lagu
─────────────────────────────────────────────────────────────────
  #  Track                               Artist                 Score
─────────────────────────────────────────────────────────────────
  1. No Good in Goodbye                  The Script            0.7986
  2. Immortals                           Fall Out Boy          0.7674
  3. Amsterdam                           Imagine Dragons       0.7562
  4. Let Her Go                          Passenger             0.7504
  5. Lay Me Down                         Sam Smith             0.7478
  6. Of The Night                        Bastille              0.7264
  7. Butterfly Fly Away                  Miley Cyrus           0.7171
  8. The Only Exception                  Paramore              0.7165
  9. Glory And Gore                      Lorde                 0.7049
 10. Say Something                       A Great

### **Deployment**

In [65]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [66]:
import pickle
import os
from scipy.sparse import save_npz

os.makedirs('recommendation_model', exist_ok=True)

# ----------------------------------------------------------
# 1. Simpan ALS model
# ----------------------------------------------------------
als_final.save('recommendation_model/als_model.npz')
print("✅ ALS model saved")

# ----------------------------------------------------------
# 2. Simpan semua objek pendukung dalam satu file
# ----------------------------------------------------------
artifacts = {
    'user_map'       : user_map,
    'item_map'       : item_map,
    'user_id_to_idx' : user_id_to_idx,
    'item_profiles'  : item_profiles,
    'audio_features' : audio_features,
    'item_id_to_name': item_id_to_name,
    'n_users'        : n_users,
    'n_items'        : n_items,
    'als_params': {
        'factors'       : 256,
        'alpha'         : 10,
        'regularization': 0.1,
        'iterations'    : 50
    },
    'metrics': {
        'ALS'   : {'Precision@10': 0.2106, 'Recall@10': 0.0537, 'NDCG@10': 0.2268},
        'Hybrid': {'Precision@10': 0.0026, 'Recall@10': 0.0263, 'NDCG@10': 0.0115}
    }
}

with open('recommendation_model/artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)
print("✅ Artifacts saved")

# ----------------------------------------------------------
# 3. Simpan train_matrix
# ----------------------------------------------------------
save_npz('recommendation_model/train_matrix.npz', train_matrix)
print("✅ Train matrix saved")

# ----------------------------------------------------------
# 4. Simpan df_cold (untuk CBF & popularity)
# ----------------------------------------------------------
df_cold.to_csv('recommendation_model/df_cold.csv', index=False)
print("✅ df_cold saved")

# ----------------------------------------------------------
# 5. Simpan user_interaction_count — GANTI df_warm
#    Hanya butuh: user_id → jumlah interaksi untuk routing
# ----------------------------------------------------------
user_interaction_count = df_warm.groupby('user_id').size().reset_index()
user_interaction_count.columns = ['user_id', 'n_interactions']
user_interaction_count.to_csv('recommendation_model/user_interaction_count.csv', index=False)
print("✅ user_interaction_count saved")

# ----------------------------------------------------------
# Cek semua file
# ----------------------------------------------------------
print("\nFile yang tersimpan:")
for f in sorted(os.listdir('recommendation_model')):
    size = os.path.getsize(f'recommendation_model/{f}') / 1024
    unit = 'KB'
    if size > 1024:
        size /= 1024
        unit = 'MB'
    print(f"  {f:<40} {size:>8.1f} {unit}")

✅ ALS model saved
✅ Artifacts saved
✅ Train matrix saved
✅ df_cold saved
✅ user_interaction_count saved

File yang tersimpan:
  als_model.npz                                 7.4 MB
  artifacts.pkl                                 2.0 MB
  df_cold.csv                                 421.9 KB
  train_matrix.npz                              4.2 MB
  user_interaction_count.csv                  549.8 KB


In [68]:
import os

# os.remove('recommendation_model/df_warm.csv')
print("🗑️ df_warm.csv dihapus")

# Cek ulang
print("\nFile yang tersimpan:")
for f in sorted(os.listdir('recommendation_model')):
    size = os.path.getsize(f'recommendation_model/{f}') / 1024
    unit = 'KB'
    if size > 1024:
        size /= 1024
        unit = 'MB'
    print(f"  {f:<40} {size:>8.1f} {unit}")

🗑️ df_warm.csv dihapus

File yang tersimpan:
  als_model.npz                                 7.4 MB
  artifacts.pkl                                 2.0 MB
  df_cold.csv                                 421.9 KB
  train_matrix.npz                              4.2 MB
  user_interaction_count.csv                  549.8 KB


In [ ]:
user_interaction_count.head()

NameError: name 'user_interaction_count' is not defined